In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

sandbox = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/sandbox_solution.csv"
)

print(train.shape)
print(sandbox.shape)

Mounted at /content/drive
(975, 30)
(5460, 2)


In [2]:
train["datetime"] = pd.to_datetime(
    train["datetime"],
    format="%d-%m-%Y %H:%M"
)

option_cols = train.columns[2:].tolist()

In [3]:
truth = dict(
    zip(
        sandbox["id"],
        sandbox["value"]
    )
)

print(len(truth))

5460


In [4]:
errors = []

for idx, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    x = np.arange(len(vals))

    valid = ~np.isnan(vals)

    if valid.sum() < 2:
        continue

    f = interp1d(
        x[valid],
        vals[valid],
        kind="linear",
        fill_value="extrapolate"
    )

    for j, col in enumerate(option_cols):

        if np.isnan(vals[j]):

            pred = float(f(j))

            key = (
                row["datetime"].strftime("%d-%m-%Y %H:%M")
                + "||"
                + col
            )

            true = truth[key]

            errors.append(
                (pred - true) ** 2
            )

print(np.mean(errors))

0.018506836175668513


In [5]:
EDGE_COLS = [
    "NIFTY27JAN2623800PE",
    "NIFTY27JAN2623900PE",
    "NIFTY27JAN2626400CE",
    "NIFTY27JAN2626500CE"
]

In [6]:
hybrid_errors = []

for idx, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    x = np.arange(len(vals))

    valid = ~np.isnan(vals)

    if valid.sum() < 2:
        continue

    linear = interp1d(
        x[valid],
        vals[valid],
        kind="linear",
        fill_value="extrapolate"
    )

    for j, col in enumerate(option_cols):

        if np.isnan(vals[j]):

            pred = float(linear(j))

            if col in EDGE_COLS:

                series = train[col]

                prev_vals = series.iloc[max(0, idx-5):idx].dropna()

                next_vals = series.iloc[idx+1:idx+6].dropna()

                neigh = pd.concat(
                    [prev_vals, next_vals]
                )

                if len(neigh) > 0:

                    time_pred = neigh.mean()

                    pred = (
                        0.7 * pred
                        + 0.3 * time_pred
                    )

            key = (
                row["datetime"].strftime("%d-%m-%Y %H:%M")
                + "||"
                + col
            )

            true = truth[key]

            hybrid_errors.append(
                (pred - true) ** 2
            )

print(np.mean(hybrid_errors))

0.016609073811896035


In [9]:
EDGE_COLS = [
    "NIFTY27JAN2623800PE",
    "NIFTY27JAN2623900PE",
    "NIFTY27JAN2624000PE",
    "NIFTY27JAN2624100PE",

    "NIFTY27JAN2626200CE",
    "NIFTY27JAN2626300CE",
    "NIFTY27JAN2626400CE",
    "NIFTY27JAN2626500CE"
]

weights = [0.9, 0.8, 0.7, 0.6, 0.5]

for w in weights:

    errs = []

    for idx, row in train.iterrows():

        vals = row[option_cols].values.astype(float)

        x = np.arange(len(vals))

        valid = ~np.isnan(vals)

        if valid.sum() < 2:
            continue

        linear = interp1d(
            x[valid],
            vals[valid],
            kind="linear",
            fill_value="extrapolate"
        )

        for j, col in enumerate(option_cols):

            if np.isnan(vals[j]):

                pred = float(linear(j))

                if col in EDGE_COLS:

                    series = train[col]

                    prev_vals = series.iloc[max(0, idx-5):idx].dropna()

                    next_vals = series.iloc[idx+1:idx+6].dropna()

                    neigh = pd.concat(
                        [prev_vals, next_vals]
                    )

                    if len(neigh) > 0:

                        time_pred = neigh.mean()

                        pred = time_pred

                key = (
                    row["datetime"].strftime("%d-%m-%Y %H:%M")
                    + "||"
                    + col
                )

                true = truth[key]

                errs.append(
                    (pred - true) ** 2
                )

    print(
        f"Linear={w:.1f}, Time={1-w:.1f}",
        np.mean(errs)
    )

Linear=0.9, Time=0.1 0.015352566956794812
Linear=0.8, Time=0.2 0.015352566956794812
Linear=0.7, Time=0.3 0.015352566956794812
Linear=0.6, Time=0.4 0.015352566956794812
Linear=0.5, Time=0.5 0.015352566956794812


In [8]:
weights = [0.4, 0.3, 0.2, 0.1, 0.0]

for w in weights:

    errs = []

    for idx, row in train.iterrows():

        vals = row[option_cols].values.astype(float)

        x = np.arange(len(vals))

        valid = ~np.isnan(vals)

        if valid.sum() < 2:
            continue

        linear = interp1d(
            x[valid],
            vals[valid],
            kind="linear",
            fill_value="extrapolate"
        )

        for j, col in enumerate(option_cols):

            if np.isnan(vals[j]):

                pred = float(linear(j))

                if col in EDGE_COLS:

                    series = train[col]

                    prev_vals = series.iloc[max(0, idx-5):idx].dropna()

                    next_vals = series.iloc[idx+1:idx+6].dropna()

                    neigh = pd.concat(
                        [prev_vals, next_vals]
                    )

                    if len(neigh) > 0:

                        time_pred = neigh.mean()

                        pred = (
                            w * pred
                            + (1-w) * time_pred
                        )

                key = (
                    row["datetime"].strftime("%d-%m-%Y %H:%M")
                    + "||"
                    + col
                )

                true = truth[key]

                errs.append(
                    (pred - true) ** 2
                )

    print(
        f"Linear={w:.1f}, Time={1-w:.1f}",
        np.mean(errs)
    )

Linear=0.4, Time=0.6 0.015325059287333001
Linear=0.3, Time=0.7 0.015033442854525203
Linear=0.2, Time=0.8 0.014810020626074005
Linear=0.1, Time=0.9 0.014654792601979402
Linear=0.0, Time=1.0 0.01456775878224141


In [10]:
EDGE_COLS = [
    "NIFTY27JAN2623800PE",
    "NIFTY27JAN2623900PE",
    "NIFTY27JAN2626400CE",
    "NIFTY27JAN2626500CE"
]

errs = []

for idx, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    x = np.arange(len(vals))

    valid = ~np.isnan(vals)

    if valid.sum() < 2:
        continue

    linear = interp1d(
        x[valid],
        vals[valid],
        kind="linear",
        fill_value="extrapolate"
    )

    for j, col in enumerate(option_cols):

        if np.isnan(vals[j]):

            pred = float(linear(j))

            if col in EDGE_COLS:

                series = train[col]

                prev_vals = series.iloc[max(0, idx-5):idx].dropna()

                next_vals = series.iloc[idx+1:idx+6].dropna()

                neigh = pd.concat(
                    [prev_vals, next_vals]
                )

                if len(neigh) > 0:

                    time_pred = neigh.mean()

                    pred = time_pred

            key = (
                row["datetime"].strftime("%d-%m-%Y %H:%M")
                + "||"
                + col
            )

            true = truth[key]

            errs.append(
                (pred - true) ** 2
            )

print("Best Edge-Time MSE =", np.mean(errs))

Best Edge-Time MSE = 0.01456775878224141


In [11]:
import os

for f in os.listdir("/content/drive/MyDrive/IV_Surface/data/raw"):
    print(f)

sandbox_solution.csv
dataset.csv


In [12]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/IV_Surface"):
    for file in files:
        print(os.path.join(root, file))

/content/drive/MyDrive/IV_Surface/data/raw/sandbox_solution.csv
/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_mean_baseline.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_best_linear_interpolation.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_cubic_interpolation.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_quadratic_interpolation.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_linear_clipped.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_time_first.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_blend_70_strike_30_time.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_blend_80_strike_20_time.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_blend_50_strike_50_time.csv
/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_local_neighbo

In [13]:
for i, cell in enumerate(get_ipython().user_ns.get('In', [])):
    print("="*50)
    print("CELL", i)
    print(cell[:500])

CELL 0

CELL 1
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

sandbox = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/sandbox_solution.csv"
)

print(train.shape)
print(sandbox.shape)
CELL 2
train["datetime"] = pd.to_datetime(
    train["datetime"],
    format="%d-%m-%Y %H:%M"
)

option_cols = train.columns[2:].tolist()
CELL 3
truth = dict(
    zip(
        sandbox["id"],
        sandbox["value"]
    )
)

print(len(truth))
CELL 4
errors = []

for idx, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    x = np.arange(len(vals))

    valid = ~np.isnan(vals)

    if valid.sum() < 2:
        continue

    f = interp1d(
        x[valid],
        vals[valid],
        kind="linear",
        fill_value="extrapolate"
    )

    for j, col in enumerate(option_cols):

        if 

In [14]:
from scipy.interpolate import interp1d
import numpy as np

errs = []

for idx, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    x = np.arange(len(vals))

    valid = ~np.isnan(vals)

    if valid.sum() < 2:
        continue

    f = interp1d(
        x[valid],
        vals[valid],
        kind="linear",
        fill_value=(
            vals[valid][0],
            vals[valid][-1]
        ),
        bounds_error=False
    )

    for j, col in enumerate(option_cols):

        if np.isnan(vals[j]):

            pred = float(f(j))

            key = (
                row["datetime"].strftime("%d-%m-%Y %H:%M")
                + "||"
                + col
            )

            true = truth[key]

            errs.append(
                (pred - true) ** 2
            )

print("Edge Clamp MSE =", np.mean(errs))

Edge Clamp MSE = 0.017838458278597924
